# Lecture 04 题目讲义：图算法与综合收束

本讲义按“建模 -> 遍历 -> MST -> 最短路 -> DAG”递进组织。每题都给出关键点与简要解法，便于课上讲解与课后复盘。

## Problem 1 · 图建模判断

**题目**：下面三个场景分别应建模成什么图？是否带权、是否有向？

1. 校园课程先修关系。
2. 校园道路导航，边权表示步行时间。
3. 社交关系“互相关注”。

**关键点**：先判断边是否有方向，再判断权值是否表示代价。

**答案提要**：
- 课程先修：有向图，通常无权，核心算法是拓扑排序。
- 道路导航：带权图；若道路可双向通行则是无向带权图。
- 互相关注：无向图；若是“单向关注”则应改成有向图。

## Problem 2 · 邻接矩阵还是邻接表

**题目**：设图有 `n=2000` 个顶点。

1. 若边数约为 `1.8e6`，更适合哪种存储？
2. 若边数约为 `6000`，更适合哪种存储？
3. 若操作是“频繁判断 (u,v) 是否有边”，你会更偏向哪种存储？

**关键点**：比较空间开销与主要操作。

**答案提要**：
- 稠密图偏向邻接矩阵。
- 稀疏图偏向邻接表。
- 高频判边通常更偏向邻接矩阵，因为查询为 `O(1)`。

## Problem 3 · DFS / BFS 手推

**题目**：给定无向图，邻接点按字母序访问。

边集：`A-B, A-C, B-D, B-E, C-F, E-G`。

1. 写出从 `A` 出发的 DFS 序列。
2. 写出从 `A` 出发的 BFS 序列。
3. BFS 中每个点的层数是多少？

**关键点**：DFS 看“深入”，BFS 看“分层”。

**答案提要**：
- DFS：`A, B, D, E, G, C, F`
- BFS：`A, B, C, D, E, F, G`
- 层数：`A:0, B/C:1, D/E/F:2, G:3`

In [ ]:
from collections import deque

graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'G'],
    'F': ['C'],
    'G': ['E'],
}

def bfs_levels(graph, start):
    dist = {start: 0}
    q = deque([start])
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in graph[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return order, dist

bfs_levels(graph, 'A')


## Problem 4 · 最小生成树执行过程

**题目**：在无向连通带权图中，边如下：

`A-B:2, A-C:3, B-C:1, B-D:4, C-D:5, D-E:2, C-E:6`

1. 用 Prim 算法从 `A` 开始构造 MST。
2. 用 Kruskal 算法构造 MST。
3. 两者得到的总权值是多少？

**关键点**：Prim 按“点集扩张”，Kruskal 按“边权排序 + 判环”。

**答案提要**：
- Prim 可依次选 `A-B, B-C, B-D, D-E`。
- Kruskal 可依次选 `B-C, A-B, D-E, B-D`。
- 总权值均为 `9`。

## Problem 5 · MST 与最短路辨析

**题目**：判断下列说法对错，并说明理由。

1. 最小生成树保证任意两点之间路径都最短。
2. Dijkstra 求出的树一定是最小生成树。
3. 最小生成树与源点无关，而最短路径树常与源点有关。

**关键点**：目标函数不同。

**答案提要**：
- 1 错：MST 只优化全局总权，不优化任意点对路径。
- 2 错：Dijkstra 优化的是从源点出发的路径长度。
- 3 对：这是两类问题最常考的区分点。

## Problem 6 · Dijkstra 手算

**题目**：给定有向图：

`S->A:2, S->B:5, A->B:1, A->C:4, B->C:1, C->T:3`

1. 用 Dijkstra 算法求从 `S` 到各点的最短距离。
2. 写出 `S` 到 `T` 的一条最短路径。

**关键点**：每一步都选当前未确定中 `dist` 最小的点，然后松弛出边。

**答案提要**：
- 距离依次可得：`A=2, B=3, C=4, T=7`。
- `S -> A -> B -> C -> T` 是最短路径。

In [ ]:
import heapq

graph = {
    'S': [('A', 2), ('B', 5)],
    'A': [('B', 1), ('C', 4)],
    'B': [('C', 1)],
    'C': [('T', 3)],
    'T': [],
}

def dijkstra(graph, start):
    dist = {v: float('inf') for v in graph}
    dist[start] = 0
    pq = [(0, start)]
    while pq:
        d, u = heapq.heappop(pq)
        if d != dist[u]:
            continue
        for v, w in graph[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist

dijkstra(graph, 'S')


## Problem 7 · Floyd 的状态理解

**题目**：说明 `dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])` 的含义。

进一步回答：为什么 Floyd 适合“所有点对最短路”，但通常不适合顶点数很大的图？

**关键点**：把 `k` 看成“允许加入的中转点”。

**答案提要**：
- 式子表示：若经过 `k` 能让 `i -> j` 更短，就更新答案。
- Floyd 会枚举所有 `k, i, j`，时间复杂度是 `O(n^3)`。
- 因而更适合点数不大、需要整张距离矩阵的时候。

## Problem 8 · 拓扑排序与判环

**题目**：有课程依赖关系：

`程序语言 -> 数据结构`

`高等数学 -> 离散数学`

`数据结构 -> 算法设计`

`离散数学 -> 算法设计`

1. 写出一个合法的拓扑序。
2. 若再增加 `算法设计 -> 程序语言`，会发生什么？

**关键点**：拓扑排序只对 DAG 有定义。

**答案提要**：
- 例如：`高等数学, 程序语言, 离散数学, 数据结构, 算法设计`。
- 若加入 `算法设计 -> 程序语言`，图中成环，不存在拓扑序。

In [ ]:
from collections import deque, defaultdict

edges = [
    ('程序语言', '数据结构'),
    ('高等数学', '离散数学'),
    ('数据结构', '算法设计'),
    ('离散数学', '算法设计'),
]

def topo_sort(edges):
    graph = defaultdict(list)
    indeg = defaultdict(int)
    nodes = set()
    for u, v in edges:
        graph[u].append(v)
        indeg[v] += 1
        nodes.add(u)
        nodes.add(v)
    q = deque(sorted([x for x in nodes if indeg[x] == 0]))
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order if len(order) == len(nodes) else None

topo_sort(edges)


## Problem 9 · 综合题型识别

**题目**：为下列场景分别匹配最合适的算法，并说明理由。

1. 无权迷宫中，求起点到终点最少走几步。
2. 铺设校内光纤，使所有楼宇连通且总成本最低。
3. 从一个仓库到所有配送点，求最短配送距离。
4. 安排一组有依赖关系的实验流程。

**关键点**：先看目标，再选算法。

**答案提要**：
- 1 用 BFS。
- 2 用 MST（Prim / Kruskal）。
- 3 用单源最短路（Dijkstra，若边权非负）。
- 4 用拓扑排序。

## 复盘清单

- 我能区分 DFS/BFS、MST、最短路、拓扑排序吗？
- 我能在 1 分钟内判断一个图题的目标吗？
- 我能手推出至少一个 Prim / Dijkstra / Topo 的过程吗？
- 我知道 visited、判环、堆、入度分别在什么算法里最关键吗？